# Fine-Tuning with Unsloth (LoRA & QLoRA)

**Unsloth** rewrites attention, RoPE, and cross-entropy kernels in Triton/CUDA,
giving roughly **2× faster training** and **~60% less VRAM** compared to stock
HuggingFace + PEFT.

This notebook covers:
- Standard **LoRA** (float16 base model)
- **QLoRA** (4-bit NF4-quantized base model + float16 LoRA adapters)
- Saving as LoRA-only, merged float16, or GGUF

**Models used:**
| Mode | Model |
|------|-------|
| LoRA | `unsloth/tinyllama-1.1B` |
| QLoRA | `unsloth/tinyllama-1.1B-bnb-4bit` |

> **GPU required.** TinyLlama 1.1B needs ~4 GB VRAM for QLoRA training.

## Environment Setup

Run once in a terminal before opening this notebook:

```bash
cd projects/unsloth_finetuning
uv sync --no-install-workspace
uv run ipython kernel install --user --env VIRTUAL_ENV ../../.venv --name=unsloth_finetuning
```

Then select the **`unsloth_finetuning`** kernel in JupyterLab.

In [ ]:
import sys
import torch

print(f'Python {sys.version}')
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB')

import unsloth, trl, datasets, transformers
print(f'unsloth=={unsloth.__version__}  trl=={trl.__version__}')
print(f'transformers=={transformers.__version__}  datasets=={datasets.__version__}')

## Load HuggingFace Token

Copy `.env.example` → `.env` in the repo root and add your token from
https://huggingface.co/settings/tokens before running this cell.

In [ ]:
import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path('../../../').resolve()))
from dotenv_config import dot_env_settings

if dot_env_settings.HF_TOKEN:
    os.environ['HF_TOKEN'] = dot_env_settings.HF_TOKEN
    print('HF_TOKEN loaded.')
else:
    print('WARNING: No HF_TOKEN. Private/gated models will fail.')

## Loading the Model: `FastLanguageModel` vs `AutoModelForCausalLM`

> **Gotcha #1 — Never use `AutoModelForCausalLM` with Unsloth.**
> Unsloth patches attention layers with custom Triton kernels. Loading via the
> standard HF API bypasses all of that — you get no speedup and the
> `for_inference()` / `for_training()` helpers won't exist.

> **Gotcha #2 — Set `max_seq_length` at load time.**
> Unsloth pre-computes RoPE caches for this length. Feeding a longer sequence
> at inference silently produces wrong positional encodings.

> **Gotcha #3 — Don't pass `device_map=`.**
> Unsloth manages device placement internally. Passing `device_map='auto'`
> conflicts with its custom kernels and causes CUDA errors.

Toggle `USE_QLORA` below to switch between standard LoRA and QLoRA.

In [ ]:
from unsloth import FastLanguageModel

USE_QLORA = True   # False = standard LoRA (float16), True = QLoRA (4-bit NF4)
MAX_SEQ_LENGTH = 512

if USE_QLORA:
    # Pre-quantized checkpoint — download is smaller (~700 MB vs ~2.2 GB)
    MODEL_NAME = 'unsloth/tinyllama-1.1B-bnb-4bit'
else:
    MODEL_NAME = 'unsloth/tinyllama-1.1B'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,            # auto-detect: bfloat16 on Ampere+, float16 otherwise
    load_in_4bit=USE_QLORA,
    # do NOT pass device_map= here
)
print(f'Loaded: {MODEL_NAME}')
print(f'dtype : {model.dtype}  |  load_in_4bit: {USE_QLORA}')

## Adding LoRA Adapters

Unsloth's `get_peft_model` is a drop-in for PEFT's version but also patches
the LoRA weight matrices with optimised Triton kernels.

> **Gotcha #4 — Only supported architectures get optimised kernels.**
> LLaMA, Mistral, Phi, Gemma, Qwen, etc. are supported. Falcon or BLOOM
> will fall back to slow HF code (or error). Check
> `unsloth.utils.supported_models` for the current list.

> **Gotcha #5 — Keep `lora_dropout=0.0` for maximum speed.**
> Non-zero dropout disables Unsloth's fused kernel path and roughly halves
> throughput. For production fine-tuning where regularisation matters,
> accept the speed hit or rely on weight decay instead.

`use_gradient_checkpointing='unsloth'` uses a smarter checkpointing schedule
than the standard HF implementation — it checkpoints fewer layers and
recomputes less, reducing overhead by ~30%.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.0,                          # 0.0 enables fused kernels
    bias='none',
    use_gradient_checkpointing='unsloth',      # smarter than standard GC
    random_state=42,
)
model.print_trainable_parameters()

## Dataset Preparation

We use the **Alpaca prompt template** from the Unsloth docs for consistency.
The format is:
```
### Instruction:
<task>

### Input:
<context>  (optional)

### Response:
<answer><EOS>
```

> **Gotcha #6 — Always append the EOS token to every training example.**
> Without it the model sees training examples as one continuous stream and
> never learns when a response is complete. At inference you'll get runaway
> generation that ignores stop conditions.

In [ ]:
from datasets import Dataset

ALPACA_PROMPT = (
    'Below is an instruction that describes a task, paired with an input that '
    'provides further context. Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{}\n\n'
    '### Input:\n{}\n\n'
    '### Response:\n{}'
)
EOS_TOKEN = tokenizer.eos_token  # append to every example

raw_examples = [
    {'instruction': 'Translate to French.', 'input': 'Hello, how are you?', 'output': 'Bonjour, comment allez-vous ?'},
    {'instruction': 'Summarise in one sentence.', 'input': 'The Eiffel Tower was built in 1889 for the World Fair in Paris and stands 330 metres tall.', 'output': 'The Eiffel Tower is a 330-metre Paris landmark built for the 1889 World Fair.'},
    {'instruction': 'Write a Python function that reverses a string.', 'input': '', 'output': 'def reverse_string(s):\n    return s[::-1]'},
    {'instruction': 'What is the capital of Japan?', 'input': '', 'output': 'The capital of Japan is Tokyo.'},
    {'instruction': 'Convert Celsius to Fahrenheit.', 'input': '100 degrees Celsius', 'output': '100°C = 212°F. Formula: F = C × 9/5 + 32'},
    {'instruction': 'Explain what a REST API is.', 'input': '', 'output': 'A REST API is an interface that lets applications communicate over HTTP using standard methods like GET, POST, PUT, and DELETE.'},
    {'instruction': 'Give three tips for better sleep.', 'input': '', 'output': '1. Keep a consistent sleep schedule. 2. Avoid screens 1 hour before bed. 3. Keep your bedroom cool and dark.'},
    {'instruction': 'Write a haiku about mountains.', 'input': '', 'output': 'Peaks pierce the grey cloud / Silence holds the ancient stone / Wind forgets no path'},
    {'instruction': 'Fix the bug in this code.', 'input': 'for i in range(10)\n    print(i)', 'output': 'for i in range(10):\n    print(i)  # Added colon and fixed indentation'},
    {'instruction': 'What is photosynthesis?', 'input': '', 'output': 'Photosynthesis is the process by which plants use sunlight, water, and CO₂ to produce glucose and oxygen.'},
]

def format_examples(batch):
    texts = [
        ALPACA_PROMPT.format(inst, inp, out) + EOS_TOKEN
        for inst, inp, out in zip(batch['instruction'], batch['input'], batch['output'])
    ]
    return {'text': texts}

dataset = Dataset.from_list(raw_examples).map(format_examples, batched=True)
print(dataset)
print('\nSample:')
print(dataset[0]['text'])

## Training with SFTTrainer

Unsloth integrates directly into TRL: `SFTTrainer` detects an Unsloth model
and automatically uses the optimised training path.

**Key settings:**
- `optim='adamw_8bit'` — 8-bit Adam saves ~1 GB VRAM with no quality loss
- `bf16` / `fp16` — choose based on GPU capability (Ampere+ supports bf16)
- `dataset_text_field='text'` — tells SFTTrainer which column to tokenize

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./unsloth_tinyllama_output',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,       # effective batch = 8
    warmup_steps=5,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field='text',
    packing=False,
    report_to='none',  # change to 'mlflow' if MLflow server is running
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print('Starting training...')
trainer.train()
print('Done.')

## Inference: Don't Forget `for_inference()`

> **Gotcha #7 — The single most-forgotten Unsloth step.**
> After training the model is still in training-kernel mode. Calling
> `.generate()` directly can produce wrong results or cryptic CUDA errors.
> Always call `FastLanguageModel.for_inference(model)` first — it switches
> to Unsloth's inference-optimised path (2× faster generation).
>
> If you **save and reload** the adapter, you do NOT need `for_inference()`
> again — it only applies to the live training object.

In [ ]:
FastLanguageModel.for_inference(model)  # switch to inference kernels

prompt = ALPACA_PROMPT.format('Write a haiku about the ocean.', '', '')
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Saving the Model

Unsloth provides three save modes:

| Mode | Method | Size | Use case |
|------|--------|------|----------|
| LoRA adapter only | `save_pretrained` | ~MB | Experiments; needs base model at load time |
| Merged float16 | `save_pretrained_merged(..., save_method='merged_16bit')` | ~2 GB | Production HF serving |
| GGUF | `save_pretrained_gguf(..., quantization_method='q4_k_m')` | ~700 MB | llama.cpp / Ollama |

> **Gotcha #8 — QLoRA merge dequantizes to float16.**
> If you loaded with `load_in_4bit=True`, the base weights are 4-bit.
> Merging LoRA into them requires dequantizing first — the result is float16,
> not 4-bit. Plan for ~2 GB of disk space for TinyLlama 1.1B merged.

In [ ]:
SAVE_PATH = './unsloth_tinyllama_lora'

# Option 1: LoRA adapter only (always works)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'LoRA adapter saved to {SAVE_PATH}')

# Option 2: merged float16 — uncomment to use
# model.save_pretrained_merged(
#     './unsloth_tinyllama_merged_16bit', tokenizer, save_method='merged_16bit'
# )

# Option 3: GGUF for llama.cpp / Ollama — uncomment to use
# model.save_pretrained_gguf(
#     './unsloth_tinyllama_gguf', tokenizer, quantization_method='q4_k_m'
# )

## Summary

### Gotcha recap

| # | Gotcha | Fix |
|---|--------|-----|
| 1 | Using `AutoModelForCausalLM` | Use `FastLanguageModel.from_pretrained` |
| 2 | `max_seq_length` inconsistent | Set once at load; reuse at inference |
| 3 | Passing `device_map=` | Omit it entirely |
| 4 | Unsupported architecture | Check `unsloth.utils.supported_models` |
| 5 | `lora_dropout > 0` | Use `0.0` for max speed; rely on weight decay |
| 6 | Missing EOS token | Append `tokenizer.eos_token` to every sample |
| 7 | Skipping `for_inference()` | Call it before any `.generate()` |
| 8 | QLoRA merge disk budget | Dequantizes to float16 — plan ~2 GB+ |

### LoRA vs QLoRA tradeoff

| | LoRA (float16) | QLoRA (4-bit NF4) |
|-|----------------|-------------------|
| Base model VRAM | ~2.2 GB | ~0.7 GB |
| Total train VRAM | ~6–8 GB | ~3–4 GB |
| Quality | Baseline | ~2–5% lower (varies) |
| Speed | Faster | Slightly slower (dequant overhead) |

**Next steps:** swap in `unsloth/llama-3.2-1b-bnb-4bit`, try a real Alpaca
dataset slice, enable `report_to='mlflow'` for experiment tracking.